# LeetCode 2181 — Merge Nodes in Between Zeros bằng Python

Link đề: https://leetcode.com/problems/merge-nodes-in-between-zeros/ · Độ khó: **Medium** · Chủ đề: Linked List, Simulation

Skeleton LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def mergeNodes(self, head: Optional[ListNode]) -> Optional[ListNode]:
        ...
```

Danh sách liên kết gồm các đoạn số dương ngăn cách bởi số 0 (đầu và cuối đều là 0). Gộp mỗi đoạn thành **một nút** mang tổng. Notebook tự cài `ListNode` + hàm dựng/xuất kiểu LeetCode, rồi đi từ **chuyển sang list Python** → **danh sách mới với nút giả (dummy)** → **tái dùng chính các nút 0 làm nút tổng (tại chỗ, `O(1)` bộ nhớ)** → **hai con trỏ ghi đè lên nút đầu đoạn** → **đệ quy theo đoạn** (và giới hạn của nó trong Python).

## 1. Mục tiêu học

1. Tự định nghĩa `ListNode`, hàm `build_list` / `to_list` để chạy thử như trên LeetCode.
2. Dùng **nút giả (dummy)** + con trỏ `tail` để xây danh sách mới gọn gàng.
3. Biến đổi **tại chỗ** (`O(1)` bộ nhớ phụ): ghi tổng vào một nút có sẵn và nối tắt (`next`) qua các nút còn lại.
4. Nhận biết lỗi kinh điển với danh sách liên kết: quên **cắt đuôi** (`tail.next = None`), tạo nút cho số 0 đầu/cuối, mất tham chiếu.
5. Hiểu vì sao lời giải **đệ quy** có thể vượt giới hạn đệ quy của Python khi danh sách dài.

## 2. Đề bài & ví dụ

Cho `head` của danh sách liên kết gồm các số nguyên, ngăn cách bởi các số `0`; nút đầu và nút cuối có giá trị `0`. Với mỗi hai số `0` liên tiếp, gộp mọi nút **nằm giữa** chúng thành **một nút** có giá trị bằng tổng. Danh sách kết quả **không** chứa số `0`. Trả về `head` mới.

**Ví dụ 1:** `[0,3,1,0,4,5,2,0]` → `[4,11]`.

**Ví dụ 2:** `[0,1,0,3,0,2,2,0]` → `[1,3,4]`.

**Ràng buộc:** số nút trong `[3, 2·10⁵]`, `0 ≤ Node.val ≤ 1000`, **không có hai số 0 liền nhau**, đầu và cuối là `0`.

In [1]:
import random
import sys
import time
from typing import List, Optional

class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

def build_list(vals):
    dummy = ListNode()
    cur = dummy
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
    return dummy.next

def to_list(head, limit=10**7):
    out = []
    while head and len(out) < limit:
        out.append(head.val); head = head.next
    return out

examples = [([0, 3, 1, 0, 4, 5, 2, 0], [4, 11]), ([0, 1, 0, 3, 0, 2, 2, 0], [1, 3, 4])]
for vals, exp in examples:
    print(vals, "-> dựng rồi xuất lại:", to_list(build_list(vals)), "| kỳ vọng sau khi gộp:", exp)

[0, 3, 1, 0, 4, 5, 2, 0] -> dựng rồi xuất lại: [0, 3, 1, 0, 4, 5, 2, 0] | kỳ vọng sau khi gộp: [4, 11]
[0, 1, 0, 3, 0, 2, 2, 0] -> dựng rồi xuất lại: [0, 1, 0, 3, 0, 2, 2, 0] | kỳ vọng sau khi gộp: [1, 3, 4]


## 3. Kiến thức nền

### 3.1 Duyệt danh sách liên kết

In [2]:
head = build_list([0, 3, 1, 0])
cur = head; i = 0
while cur:
    print(f"  nút {i}: val = {cur.val}, next là {'None' if cur.next is None else cur.next.val}")
    cur = cur.next; i += 1

  nút 0: val = 0, next là 3
  nút 1: val = 3, next là 1
  nút 2: val = 1, next là 0
  nút 3: val = 0, next là None


### 3.2 Nút giả (dummy) + con trỏ đuôi

`dummy` là nút "đứng trước đầu"; mọi lần thêm nút chỉ là `tail.next = node; tail = node`. Cuối cùng trả `dummy.next` — không cần xử lý riêng nút đầu tiên.

In [3]:
dummy = ListNode(); tail = dummy
for v in [7, 8, 9]:
    tail.next = ListNode(v); tail = tail.next
print(to_list(dummy.next))

[7, 8, 9]


### 3.3 Nối tắt (bỏ qua nút) và cắt đuôi

`a.next = b` làm mọi nút giữa `a` và `b` "biến mất" khỏi danh sách (không còn đường đi tới). Khi biến đổi tại chỗ, nút cuối của kết quả phải có `next = None`, nếu không danh sách sẽ "kéo theo" phần thừa.

In [4]:
h = build_list([1, 2, 3, 4, 5])
h.next = h.next.next.next            # 1 -> 4
print("nối tắt:", to_list(h))
h.next.next = None                   # cắt sau 4
print("cắt đuôi:", to_list(h))

nối tắt: [1, 4, 5]
cắt đuôi: [1, 4]


### 3.4 Giới hạn đệ quy của Python

In [5]:
print("giới hạn đệ quy mặc định:", sys.getrecursionlimit(), "| số đoạn tối đa theo ràng buộc: ~", (2 * 10**5 - 1) // 2)

giới hạn đệ quy mặc định: 1000 | số đoạn tối đa theo ràng buộc: ~ 99999


## 4. Ý tưởng nền tảng / trực giác

Đọc danh sách từ trái sang phải với một biến `s` (tổng đoạn hiện tại):
- gặp số **khác 0** ⇒ cộng vào `s`;
- gặp số **0** (trừ số 0 đầu tiên) ⇒ một đoạn vừa kết thúc ⇒ xuất `s`, đặt lại `s = 0`.

Tính tay ví dụ 1 `[0,3,1,0,4,5,2,0]`:

| nút | 0 | 3 | 1 | 0 | 4 | 5 | 2 | 0 |
|---|---|---|---|---|---|---|---|---|
| `s` sau nút | 0 | 3 | 4 | → xuất **4**, 0 | 4 | 9 | 11 | → xuất **11**, 0 |

Kết quả `[4, 11]` ✓. Vì không có hai số 0 liền nhau, mỗi đoạn khác rỗng và có tổng `> 0`?… **Chưa chắc**: các giá trị trong đoạn là `Node.val ≥ 0` và **khác 0** (vì 0 là dấu ngăn) ⇒ mỗi đoạn có ít nhất một số dương ⇒ tổng `≥ 1` ⇒ kết quả quả thật không chứa 0.

## 5. Cách 1 — Brute Force / baseline: chuyển sang list Python

Đọc hết giá trị vào list, tách theo số 0, tính tổng từng đoạn, dựng danh sách mới. Đơn giản, dễ thấy đúng — dùng làm **chuẩn đối chiếu**.

In [6]:
def merge_via_list(head):
    vals = to_list(head)
    sums, s = [], 0
    for v in vals[1:]:                 # bỏ số 0 đầu tiên
        if v == 0:
            sums.append(s); s = 0
        else:
            s += v
    return build_list(sums)

for vals, exp in examples:
    print(to_list(merge_via_list(build_list(vals))), "(kỳ vọng", exp, ")")

[4, 11] (kỳ vọng [4, 11] )
[1, 3, 4] (kỳ vọng [1, 3, 4] )


**Độ phức tạp:** `O(n)` thời gian, `O(n)` bộ nhớ phụ (list giá trị + danh sách mới). **Nút thắt:** tốn bộ nhớ gấp đôi; đề cho phép sửa trực tiếp danh sách ⇒ tận dụng các nút có sẵn.

## 6. Ý tưởng thuật toán / bước đột phá

**Quan sát:** mỗi đoạn được **mở đầu** bởi đúng một số 0 (số 0 đứng trước nó). Có bao nhiêu đoạn thì có bấy nhiêu số 0 "mở đầu" (mọi số 0 trừ số 0 cuối cùng). ⇒ Có thể **tái dùng** nút 0 mở đầu làm nút tổng của đoạn: ghi tổng vào `zero.val`, nối `zero.next` tới nút 0 mở đầu đoạn tiếp theo; nút 0 mở đầu cuối cùng (trước số 0 cuối) có `next = None`.

**Quy tắc (tại chỗ):** con trỏ `z` ở một nút 0 mở đầu; quét `cur = z.next` cộng dồn tới khi gặp số 0 tiếp theo `nz`:
- **nếu** `nz.next` là `None` (đó là số 0 cuối) **thì** `z.val = s`, `z.next = None`, kết thúc;
- **ngược lại** `z.val = s`, `z.next = nz`, chuyển `z = nz`.

Đầu kết quả chính là `head` (nút 0 đầu tiên, giờ mang tổng đoạn đầu).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Danh sách mới với nút giả (một lượt)

In [7]:
def merge_dummy(head):
    dummy = ListNode(); tail = dummy
    s = 0
    cur = head.next                    # bỏ số 0 đầu
    while cur:
        if cur.val == 0:
            tail.next = ListNode(s); tail = tail.next
            s = 0
        else:
            s += cur.val
        cur = cur.next
    return dummy.next

for vals, exp in examples:
    print(to_list(merge_dummy(build_list(vals))), "(kỳ vọng", exp, ")")

[4, 11] (kỳ vọng [4, 11] )
[1, 3, 4] (kỳ vọng [1, 3, 4] )


**Độ phức tạp:** `O(n)` thời gian, `O(số đoạn)` nút mới. Đây là lời giải "chuẩn" dễ viết nhất.

### 7.2 Cách 3 — Tại chỗ: tái dùng nút 0 mở đầu làm nút tổng

In [8]:
def merge_reuse_zeros(head):
    z = head                           # nút 0 mở đầu đoạn hiện tại
    while True:
        s = 0
        cur = z.next
        while cur.val != 0:
            s += cur.val
            cur = cur.next
        z.val = s                      # cur là số 0 kết thúc đoạn
        if cur.next is None:           # đó là số 0 cuối cùng
            z.next = None
            return head
        z.next = cur                   # cur trở thành nút 0 mở đầu đoạn tiếp theo
        z = cur

for vals, exp in examples:
    print(to_list(merge_reuse_zeros(build_list(vals))), "(kỳ vọng", exp, ")")

[4, 11] (kỳ vọng [4, 11] )
[1, 3, 4] (kỳ vọng [1, 3, 4] )


**Độ phức tạp:** `O(n)` thời gian, `O(1)` bộ nhớ phụ. Mọi nút số dương bị "bỏ rơi" (không còn ai trỏ tới) — Python tự thu gom.

### 7.3 Cách 4 — Hai con trỏ: ghi tổng đè lên các nút từ đầu danh sách

Con trỏ `write` bắt đầu ở `head.next`… cách khác dễ nhớ: `write` chạy trên **chính danh sách**, mỗi khi kết thúc một đoạn thì ghi tổng vào `write.val` và tiến `write` một bước; cuối cùng cắt sau nút `write` cuối cùng. Vì số đoạn `≤` số nút, `write` không bao giờ vượt `read`.

In [9]:
def merge_two_pointers(head):
    write = head                       # nút sẽ nhận tổng tiếp theo
    last = None                        # nút cuối cùng đã nhận tổng
    read = head.next
    s = 0
    while read:
        if read.val == 0:
            write.val = s
            last = write
            write = write.next
            s = 0
        else:
            s += read.val
        read = read.next
    last.next = None                   # cắt phần thừa phía sau
    return head

for vals, exp in examples:
    print(to_list(merge_two_pointers(build_list(vals))), "(kỳ vọng", exp, ")")

[4, 11] (kỳ vọng [4, 11] )
[1, 3, 4] (kỳ vọng [1, 3, 4] )


**Vì sao `write` luôn đi sau `read`?** Khi kết thúc đoạn thứ `t`, `read` đã qua ít nhất `2t` nút (mỗi đoạn ≥ 1 số + 1 số 0), còn `write` mới tiến `t` bước ⇒ ghi đè chỉ chạm các nút `read` **đã đọc**. **Độ phức tạp:** `O(n)`, `O(1)`.

### 7.4 Cách 5 — Đệ quy theo đoạn (chỉ an toàn khi ít đoạn)

`merge(z)`: tính tổng đoạn sau nút 0 `z`, tới số 0 kế `nz`; nếu `nz` là số 0 cuối thì trả nút tổng; ngược lại `node.next = merge(nz)`. Độ sâu đệ quy = **số đoạn** — tới ~10⁵ với ràng buộc của đề, vượt giới hạn 1000 của Python ⇒ chỉ minh họa, không nộp.

In [10]:
def merge_recursive(z):
    s = 0
    cur = z.next
    while cur.val != 0:
        s += cur.val; cur = cur.next
    node = ListNode(s)
    if cur.next is not None:
        node.next = merge_recursive(cur)
    return node

for vals, exp in examples:
    print(to_list(merge_recursive(build_list(vals))), "(kỳ vọng", exp, ")")

def segments(vals):
    return vals.count(0) - 1

worst = [0] + [1, 0] * (10**5 - 1)
print("danh sách xấu nhất có", len(worst), "nút,", segments(worst), "đoạn -> độ sâu đệ quy", segments(worst), "> giới hạn", sys.getrecursionlimit())

[4, 11] (kỳ vọng [4, 11] )
[1, 3, 4] (kỳ vọng [1, 3, 4] )
danh sách xấu nhất có 199999 nút, 99999 đoạn -> độ sâu đệ quy 99999 > giới hạn 1000


## 8. TRACE từng bước

### 8.1 Trace cách 3 (tái dùng nút 0) trên ví dụ 1

In [11]:
def trace_reuse(vals):
    head = build_list(vals)
    nodes = []
    c = head
    while c:
        nodes.append(c); c = c.next
    idx = {id(n): i for i, n in enumerate(nodes)}
    z = head
    while True:
        s = 0; cur = z.next; seg = []
        while cur.val != 0:
            s += cur.val; seg.append(cur.val); cur = cur.next
        z.val = s
        if cur.next is None:
            z.next = None
            print(f"  nút 0 #{idx[id(z)]}: đoạn {seg} -> ghi {s}, đây là đoạn cuối -> next = None")
            break
        z.next = cur
        print(f"  nút 0 #{idx[id(z)]}: đoạn {seg} -> ghi {s}, nối tới nút 0 #{idx[id(cur)]}")
        z = cur
    print("  => kết quả:", to_list(head))
    return to_list(head)

_ = trace_reuse([0, 3, 1, 0, 4, 5, 2, 0])

  nút 0 #0: đoạn [3, 1] -> ghi 4, nối tới nút 0 #3
  nút 0 #3: đoạn [4, 5, 2] -> ghi 11, đây là đoạn cuối -> next = None
  => kết quả: [4, 11]


### 8.2 Trace cách 4 (hai con trỏ) trên ví dụ 2

In [12]:
def trace_two_pointers(vals):
    head = build_list(vals)
    write, last, read, s = head, None, head.next, 0
    step = 0
    while read:
        step += 1
        if read.val == 0:
            write.val = s; last = write; write = write.next
            print(f"  bước {step}: đọc 0 -> ghi tổng {s} | danh sách hiện tại (10 nút đầu): {to_list(head, 10)}")
            s = 0
        else:
            s += read.val
            print(f"  bước {step}: đọc {read.val} -> s = {s}")
        read = read.next
    last.next = None
    print("  => sau khi cắt đuôi:", to_list(head))

trace_two_pointers([0, 1, 0, 3, 0, 2, 2, 0])

  bước 1: đọc 1 -> s = 1
  bước 2: đọc 0 -> ghi tổng 1 | danh sách hiện tại (10 nút đầu): [1, 1, 0, 3, 0, 2, 2, 0]
  bước 3: đọc 3 -> s = 3
  bước 4: đọc 0 -> ghi tổng 3 | danh sách hiện tại (10 nút đầu): [1, 3, 0, 3, 0, 2, 2, 0]
  bước 5: đọc 2 -> s = 2
  bước 6: đọc 2 -> s = 4
  bước 7: đọc 0 -> ghi tổng 4 | danh sách hiện tại (10 nút đầu): [1, 3, 4, 3, 0, 2, 2, 0]
  => sau khi cắt đuôi: [1, 3, 4]


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def mergeNodes(self, head: Optional[ListNode]) -> Optional[ListNode]:
        z = head                                # nút 0 mở đầu đoạn -> sẽ mang tổng đoạn
        while True:
            s = 0
            cur = z.next
            while cur.val != 0:
                s += cur.val
                cur = cur.next
            z.val = s
            if cur.next is None:                # cur là số 0 cuối cùng
                z.next = None
                return head
            z.next = cur
            z = cur

for vals, exp in examples:
    assert to_list(Solution().mergeNodes(build_list(vals))) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

Gọi các số 0 theo thứ tự là `Z₀, Z₁, …, Z_k` (`Z₀` là `head`, `Z_k` là nút cuối); đoạn `i` (`1 ≤ i ≤ k`) là các nút giữa `Z_{i−1}` và `Z_i`, khác rỗng vì không có hai số 0 liền nhau.

**Bất biến của cách 3:** ở đầu vòng lặp thứ `i`, `z = Z_{i−1}`, và các nút `Z₀, …, Z_{i−2}` đã mang đúng tổng các đoạn `1, …, i−1`, nối với nhau theo thứ tự, `Z_{i−2}.next = Z_{i−1}`.
- Vòng trong cộng đúng các nút của đoạn `i` (dừng ở số 0 đầu tiên sau `z`, tức `Z_i`).
- Ghi `Z_{i−1}.val = tổng đoạn i`. Nếu `Z_i` là nút cuối (`i = k`), đặt `next = None` ⇒ danh sách kết quả là `Z₀ → … → Z_{k−1}` mang `k` tổng, không còn số 0.
- Ngược lại nối `Z_{i−1}.next = Z_i`, sang vòng `i + 1` — bất biến được giữ.

Mỗi tổng `≥ 1` vì đoạn khác rỗng và các phần tử trong đoạn là số **khác 0**, không âm.

**Cách 4:** `write` đi qua đúng `k` nút đầu tiên của danh sách gốc, ghi lần lượt các tổng; `write` không vượt `read` (mục 7.3) nên không ghi đè dữ liệu chưa đọc; cắt sau nút thứ `k`.

## 11. Phân tích Time / Space Complexity

Gọi `n` = số nút, `k` = số đoạn (`≤ (n − 1)/2`).

| Cách | Time | Space phụ |
|---|---|---|
| 1. Qua list Python | `O(n)` | `O(n)` |
| 2. Danh sách mới + dummy | `O(n)` | `O(k)` nút mới |
| 3. Tái dùng nút 0 | `O(n)` | `O(1)` |
| 4. Hai con trỏ ghi đè | `O(n)` | `O(1)` |
| 5. Đệ quy theo đoạn | `O(n)` | `O(k)` ngăn xếp — vượt giới hạn khi `k` lớn |

In [14]:
rnd = random.Random(2181)
def random_case(n_segments, rnd, maxlen=5):
    vals = [0]
    for _ in range(n_segments):
        vals += [rnd.randint(1, 1000) for _ in range(rnd.randint(1, maxlen))] + [0]
    return vals

big = random_case(40000, rnd)
print("số nút:", len(big))
for name, fn in [("via_list", merge_via_list), ("dummy", merge_dummy), ("reuse_zeros", merge_reuse_zeros),
                 ("two_pointers", merge_two_pointers), ("Solution", Solution().mergeNodes)]:
    h = build_list(big)
    t0 = time.perf_counter(); r = fn(h); dt = time.perf_counter() - t0
    print(f"{name:<13}: {len(to_list(r))} nút kết quả, {dt * 1000:7.1f} ms")

số nút: 159434


via_list     : 40000 nút kết quả,    73.6 ms


dummy        : 40000 nút kết quả,    17.1 ms


reuse_zeros  : 40000 nút kết quả,     9.9 ms


two_pointers : 40000 nút kết quả,    15.8 ms
Solution     : 40000 nút kết quả,    10.6 ms


## 12. Bộ test

In [15]:
ALL = {
    "dummy": merge_dummy,
    "reuse_zeros": merge_reuse_zeros,
    "two_pointers": merge_two_pointers,
    "recursive": merge_recursive,
    "Solution": lambda h: Solution().mergeNodes(h),
}

def expected(vals):
    out, s = [], 0
    for v in vals[1:]:
        if v == 0: out.append(s); s = 0
        else: s += v
    return out

def check(vals, exp=None, skip=()):
    ref = to_list(merge_via_list(build_list(vals)))
    assert ref == expected(vals) and 0 not in ref
    for name, fn in ALL.items():
        if name in skip:
            continue
        got = to_list(fn(build_list(vals)))
        assert got == ref, (name, vals, got, ref)
    if exp is not None:
        assert ref == exp
    return ref

for vals, exp in examples:
    check(vals, exp)
print("Ví dụ đề: OK")

check([0, 5, 0], [5])                          # nhỏ nhất: 3 nút
check([0, 1000, 0, 1000, 0], [1000, 1000])
check([0, 1, 1, 1, 1, 1, 0], [5])
check([0] + [1, 0] * 6, [1] * 6)               # nhiều đoạn một phần tử
print("Edge case: OK")

rnd = random.Random(2021)
for _ in range(2000):
    check(random_case(rnd.randint(1, 8), rnd))
print("2000 danh sách ngẫu nhiên nhỏ: OK")

for _ in range(5):
    check(random_case(rnd.randint(1000, 60000), rnd), skip=("recursive",))
print("5 danh sách lớn (bỏ bản đệ quy): OK")
check(worst, skip=("recursive",))
print("Danh sách xấu nhất ~2e5 nút: OK")

Ví dụ đề: OK
Edge case: OK


2000 danh sách ngẫu nhiên nhỏ: OK


5 danh sách lớn (bỏ bản đệ quy): OK


Danh sách xấu nhất ~2e5 nút: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Tạo nút cho **mọi** số 0, kể cả số 0 đầu tiên

In [16]:
def bug_every_zero(head):
    dummy = ListNode(); tail = dummy; s = 0
    cur = head                                   # SAI: không bỏ qua số 0 đầu
    while cur:
        if cur.val == 0:
            tail.next = ListNode(s); tail = tail.next; s = 0
        else:
            s += cur.val
        cur = cur.next
    return dummy.next

vals = [0, 3, 1, 0, 4, 5, 2, 0]
print("sai :", to_list(bug_every_zero(build_list(vals))), "| đúng:", to_list(Solution().mergeNodes(build_list(vals))))

sai : [0, 4, 11] | đúng: [4, 11]


**Vì sao sai:** số 0 đầu tiên "kết thúc" một đoạn rỗng ⇒ thêm nút `0` vào đầu kết quả. **Sửa:** bắt đầu từ `head.next`.

### Lỗi 2 — Biến đổi tại chỗ nhưng quên cắt đuôi

In [17]:
def bug_no_cut(head):
    write, read, s = head, head.next, 0
    while read:
        if read.val == 0:
            write.val = s; write = write.next; s = 0
        else:
            s += read.val
        read = read.next
    return head                                   # SAI: thiếu last.next = None

vals = [0, 3, 1, 0, 4, 5, 2, 0]
print("sai :", to_list(bug_no_cut(build_list(vals))), "| đúng:", to_list(Solution().mergeNodes(build_list(vals))))

sai : [4, 11, 1, 0, 4, 5, 2, 0] | đúng: [4, 11]


**Vì sao sai:** chỉ `k = 2` nút đầu được ghi tổng, nhưng phần còn lại của danh sách gốc vẫn được nối phía sau. **Sửa:** nhớ nút cuối đã ghi và đặt `last.next = None`.

### Lỗi 3 — Quên đặt lại tổng `s = 0` sau mỗi đoạn

In [18]:
def bug_no_reset(head):
    dummy = ListNode(); tail = dummy; s = 0
    cur = head.next
    while cur:
        if cur.val == 0:
            tail.next = ListNode(s); tail = tail.next            # SAI: thiếu s = 0
        else:
            s += cur.val
        cur = cur.next
    return dummy.next

vals = [0, 1, 0, 3, 0, 2, 2, 0]
print("sai :", to_list(bug_no_reset(build_list(vals))), "(tổng tiền tố!) | đúng:", to_list(Solution().mergeNodes(build_list(vals))))

sai : [1, 4, 8] (tổng tiền tố!) | đúng: [1, 3, 4]


**Sửa:** `s = 0` ngay sau khi xuất tổng một đoạn.

### Lỗi 4 — Cách 3 nhưng nối `z.next = cur.next` (bỏ qua luôn nút 0 mở đầu đoạn sau)

In [19]:
def bug_skip_next_zero(head):
    z = head
    while True:
        s = 0; cur = z.next
        while cur.val != 0:
            s += cur.val; cur = cur.next
        z.val = s
        if cur.next is None:
            z.next = None; return head
        z.next = cur.next                         # SAI: phải nối tới chính nút 0 'cur'
        z = cur.next                              #      và z phải là nút 0, không phải nút số

vals = [0, 3, 1, 0, 4, 5, 2, 0]
print("sai :", to_list(bug_skip_next_zero(build_list(vals))), "| đúng:", to_list(Solution().mergeNodes(build_list(vals))))

sai : [4, 7] | đúng: [4, 11]


**Vì sao sai:** `z` trở thành nút **số** đầu đoạn sau; vòng trong bắt đầu từ `z.next` nên **bỏ mất** số đầu tiên của đoạn (`4`), rồi ghi đè `z.val`. **Sửa:** `z.next = cur; z = cur` — nút 0 kết thúc đoạn này là nút 0 mở đầu đoạn sau.

### Lỗi 5 — Dùng đệ quy cho danh sách dài

In [20]:
def recursion_depth_needed(vals):
    return vals.count(0) - 1

for vals in [examples[0][0], random_case(500, random.Random(1)), worst]:
    need = recursion_depth_needed(vals)
    print(f"  {len(vals):>6} nút: cần độ sâu {need:>6} -> {'an toàn' if need < sys.getrecursionlimit() - 50 else 'VƯỢT giới hạn (RecursionError nếu chạy)'}")

       8 nút: cần độ sâu      2 -> an toàn
    2005 nút: cần độ sâu    500 -> an toàn
  199999 nút: cần độ sâu  99999 -> VƯỢT giới hạn (RecursionError nếu chạy)


**Vì sao là lỗi:** bản đệ quy (cách 5) đúng về logic nhưng với `~10⁵` đoạn sẽ ném `RecursionError` (ta không chạy để giữ notebook sạch, chỉ tính độ sâu cần thiết). Tăng `sys.setrecursionlimit` cũng có thể làm tràn ngăn xếp C. **Sửa:** viết vòng lặp (cách 2–4).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Danh sách liên kết + "tạo danh sách mới" ⇒ **dummy + tail**.
- "Gộp đoạn giữa các dấu ngăn" ⇒ một biến tích lũy, xuất khi gặp dấu ngăn, **đặt lại** sau khi xuất.
- Biến đổi tại chỗ ⇒ luôn tự hỏi: nút cuối có `next = None` chưa?
- Tái dùng nút dấu ngăn làm nút kết quả là mẹo `O(1)` bộ nhớ hay gặp.
- Python: tránh đệ quy trên danh sách dài; tự viết `build_list`/`to_list` để test nhanh.

## 15. Biến thể của bài

1. **Dấu ngăn khác 0** (ví dụ `-1`), hoặc đầu/cuối **không** đảm bảo là dấu ngăn — cần xử lý đoạn rỗng/đoạn mở (bài tập 5).
2. **Gộp thành tích / max / độ dài đoạn** thay vì tổng — chỉ đổi phép tích lũy.
3. **LeetCode 1171 — Remove Zero Sum Consecutive Nodes from Linked List:** tiền tố + hashmap trên danh sách liên kết.
4. **LeetCode 2058 — Find the Minimum and Maximum Number of Nodes Between Critical Points.**
5. **Cùng bài trên mảng:** `[sum(g) for g in tách theo 0]` — `itertools.groupby`.

In [21]:
from itertools import groupby

def merge_array_groupby(vals):
    """Biến thể 5: cùng bài trên mảng, một dòng với groupby."""
    return [sum(g) for is_zero, g in groupby(vals, key=lambda v: v == 0) if not is_zero]

def merge_nodes_by(head, op, init):
    """Biến thể 2: gộp bằng phép tùy ý (tích, max, đếm...)."""
    dummy = ListNode(); tail = dummy; acc = init
    cur = head.next
    while cur:
        if cur.val == 0:
            tail.next = ListNode(acc); tail = tail.next; acc = init
        else:
            acc = op(acc, cur.val)
        cur = cur.next
    return dummy.next

vals = [0, 3, 1, 0, 4, 5, 2, 0]
print("groupby trên mảng:", merge_array_groupby(vals))
print("tích:", to_list(merge_nodes_by(build_list(vals), lambda a, b: a * b, 1)),
      "| max:", to_list(merge_nodes_by(build_list(vals), max, 0)),
      "| độ dài đoạn:", to_list(merge_nodes_by(build_list(vals), lambda a, b: a + 1, 0)))
rnd_v = random.Random(15)
for _ in range(500):
    v = random_case(rnd_v.randint(1, 6), rnd_v)
    assert merge_array_groupby(v) == expected(v)
print("groupby khớp kỳ vọng trên 500 mảng")

groupby trên mảng: [4, 11]
tích: [3, 40] | max: [3, 5] | độ dài đoạn: [2, 3]
groupby khớp kỳ vọng trên 500 mảng


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space phụ | Khi nào dùng |
|---|---|---|---|---|
| 1. Qua list | tách mảng | `O(n)` | `O(n)` | kiểm thử |
| 2. Dummy + nút mới | tích lũy, xuất ở số 0 | `O(n)` | `O(k)` | dễ viết nhất |
| 3. Tái dùng nút 0 | nút 0 mang tổng | `O(n)` | `O(1)` | **nộp bài** |
| 4. Hai con trỏ | ghi đè từ đầu | `O(n)` | `O(1)` | luyện "read/write pointer" |
| 5. Đệ quy | theo đoạn | `O(n)` | `O(k)` stack | chỉ khi ít đoạn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `[0, 2, 0, 2, 2, 0, 2, 2, 2, 0]` sau khi gộp?

**Bài 2 (tính tay):** Với cách 4 trên `[0, 5, 6, 0, 7, 0]`, sau khi xử lý xong, giá trị các nút của danh sách gốc (trước khi cắt) là gì? `last` trỏ nút thứ mấy?

**Bài 3 (điền TODO):** Hoàn thiện `merge_max(head)` — mỗi đoạn thay bằng **giá trị lớn nhất** của đoạn, tại chỗ.

**Bài 4 (tự viết lại):** Viết lại cách 2 nhưng trả về **cả** danh sách kết quả **và** danh sách độ dài các đoạn (dạng list Python).

**Bài 5 (mini challenge):** Bỏ giả thiết "đầu và cuối là 0" và "không có hai số 0 liền nhau": gộp các đoạn **khác rỗng** giữa các số 0 (đoạn trước số 0 đầu tiên và sau số 0 cuối cùng cũng tính). Ví dụ `[3, 0, 0, 1, 2]` → `[3, 3]`. Kiểm bằng `groupby`.

In [22]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def merge_max(head):
    # TODO: như Solution nhưng m = max(m, cur.val)
    pass

# Bài 4 — TODO
def merge_with_lengths(head):
    # TODO: tích lũy s và len
    pass

# Bài 5 — TODO
def merge_general(head):
    # TODO: dummy + tail; xuất khi gặp 0 hoặc hết danh sách, chỉ khi đoạn khác rỗng
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Ba đoạn: `[2]`, `[2,2]`, `[2,2,2]`.
- Mức 2: Tổng `2, 4, 6`.
- Mức 3: `[2, 4, 6]`.

**Bài 2**
- Mức 1: Có 2 đoạn ⇒ `write` ghi vào nút 0 và nút 1.
- Mức 2: Nút 0 ← 11, nút 1 ← 7.
- Mức 3: Trước khi cắt: `[11, 7, 6, 0, 7, 0]`, `last` là nút thứ 1 (đếm từ 0).

**Bài 3**
- Mức 1: Thay `s += cur.val` bằng `m = max(m, cur.val)`.
- Mức 2: Khởi tạo `m = 0` (mọi giá trị trong đoạn ≥ 1).
- Mức 3: Kiểm bằng `merge_nodes_by(head, max, 0)`.

**Bài 4**
- Mức 1: Thêm biến `length`.
- Mức 2: Xuất `(s, length)` khi gặp số 0.
- Mức 3: So với `groupby` trên mảng.

**Bài 5**
- Mức 1: Duyệt **mọi** nút, không bỏ nút đầu.
- Mức 2: Dùng cờ `in_seg` (đoạn khác rỗng) thay vì so `s > 0` (vì đề tổng quát có thể có số dương… nếu chỉ có số dương thì `s > 0` cũng được).
- Mức 3: Sau vòng lặp, nếu `in_seg` thì xuất đoạn cuối.

## 19. Đáp án tham khảo cho bài tập

In [23]:
# Bài 1
r1 = to_list(Solution().mergeNodes(build_list([0, 2, 0, 2, 2, 0, 2, 2, 2, 0])))
print("Bài 1:", r1); assert r1 == [2, 4, 6]

# Bài 2
h2 = build_list([0, 5, 6, 0, 7, 0])
write, last, read, s = h2, None, h2.next, 0
while read:
    if read.val == 0:
        write.val = s; last = write; write = write.next; s = 0
    else:
        s += read.val
    read = read.next
before_cut = to_list(h2)
last_index = 0; c = h2
while c is not last:
    c = c.next; last_index += 1
print("Bài 2: trước khi cắt", before_cut, "| last là nút thứ", last_index)
assert before_cut == [11, 7, 6, 0, 7, 0] and last_index == 1

# Bài 3
def merge_max_sol(head):
    z = head
    while True:
        m = 0; cur = z.next
        while cur.val != 0:
            m = max(m, cur.val); cur = cur.next
        z.val = m
        if cur.next is None:
            z.next = None; return head
        z.next = cur; z = cur

rnd = random.Random(3)
for _ in range(500):
    v = random_case(rnd.randint(1, 6), rnd)
    assert to_list(merge_max_sol(build_list(v))) == to_list(merge_nodes_by(build_list(v), max, 0))
print("Bài 3 OK, ví dụ:", to_list(merge_max_sol(build_list([0, 3, 1, 0, 4, 5, 2, 0]))))

Bài 1: [2, 4, 6]
Bài 2: trước khi cắt [11, 7, 6, 0, 7, 0] | last là nút thứ 1
Bài 3 OK, ví dụ: [3, 5]


In [24]:
# Bài 4
def merge_with_lengths_sol(head):
    dummy = ListNode(); tail = dummy; s = length = 0; lens = []
    cur = head.next
    while cur:
        if cur.val == 0:
            tail.next = ListNode(s); tail = tail.next; lens.append(length); s = length = 0
        else:
            s += cur.val; length += 1
        cur = cur.next
    return dummy.next, lens

# Bài 5
def merge_general_sol(head):
    dummy = ListNode(); tail = dummy; s = 0; in_seg = False
    cur = head
    while cur:
        if cur.val == 0:
            if in_seg:
                tail.next = ListNode(s); tail = tail.next
            s = 0; in_seg = False
        else:
            s += cur.val; in_seg = True
        cur = cur.next
    if in_seg:
        tail.next = ListNode(s)
    return dummy.next

rnd = random.Random(5)
for _ in range(1500):
    v = random_case(rnd.randint(1, 6), rnd)
    h, lens = merge_with_lengths_sol(build_list(v))
    assert to_list(h) == expected(v) and lens == [len(list(g)) for z, g in groupby(v, key=lambda x: x == 0) if not z]
    w = [rnd.choice([0, 0, rnd.randint(1, 9)]) for _ in range(rnd.randint(1, 12))]
    assert to_list(merge_general_sol(build_list(w))) == merge_array_groupby(w)
print("Bài 4, Bài 5 OK; ví dụ:", merge_with_lengths_sol(build_list([0, 3, 1, 0, 4, 5, 2, 0]))[1],
      to_list(merge_general_sol(build_list([3, 0, 0, 1, 2]))))

Bài 4, Bài 5 OK; ví dụ: [2, 3] [3, 3]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template tại chỗ (tái dùng nút 0):**

```python
z = head
while True:
    s, cur = 0, z.next
    while cur.val != 0:
        s += cur.val; cur = cur.next
    z.val = s
    if cur.next is None:
        z.next = None; return head
    z.next = cur; z = cur
```

**Template dummy:** `dummy = ListNode(); tail = dummy; … tail.next = ListNode(s); tail = tail.next; … return dummy.next`.

**Checklist:**
- [ ] Bỏ qua số 0 **đầu tiên**?
- [ ] Đặt lại `s = 0` sau mỗi đoạn?
- [ ] Biến đổi tại chỗ: nút cuối có `next = None`?
- [ ] Nối tới **nút 0** kết thúc đoạn (không phải nút sau nó)?
- [ ] Không dùng đệ quy cho danh sách dài?
- [ ] Chữ ký đúng: `def mergeNodes(self, head: Optional[ListNode]) -> Optional[ListNode]`?